# Notebook 07: Explainability and Machine Learning vs Deep Learning Comparison

## Overview
This is the final notebook of the machine learning pipeline. It does two things. First, it opens up the best model and explains which feature families and individual features drive the predictions, using native feature importance, permutation importance, and SHAP values. This is the part of the machine learning pipeline that the deep learning pipeline cannot offer as directly: readable explanations of why a prediction was made. Second, it builds a comparison table of all four machine learning models, then places the best one alongside the deep learning results from the companion ResNet-50 project so the two pipelines can be discussed together.

## Objectives
1. Load the best model and compute native feature importance grouped by family.
2. Compute permutation importance on the test set as a model-agnostic cross-check.
3. Generate SHAP values with TreeExplainer and summarise them per feature family and per class.
4. Build a leaderboard table from the saved metric files for all four ML models.
5. Compare the best ML model against the ResNet-50 result from the companion deep learning pipeline.

## Note on the deep learning comparison
The ResNet-50 result used below (test accuracy 99.87 percent) comes from the companion project
at https://github.com/hodyek/lung-colon-cancer-histopathology, submitted as prior work by
Odyek Henry (2025/HD07/26018U). The same LC25000 dataset and the same stratified 70/15/15
split with seed 42 were used in both pipelines, so the comparison reflects method differences
rather than data differences.

In [ ]:
# ----- Environment setup -----
!pip install -q scikit-image seaborn joblib xgboost shap

from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ---- Clone the project repository so notebooks can import from src/ ----
REPO       = "lung-colon-cancer-histopathology-ml"
REPO_LOCAL = "/content/" + REPO
REPO_URL   = "https://github.com/hodyek/" + REPO + ".git"

if not os.path.exists(REPO_LOCAL):
    !git clone $REPO_URL $REPO_LOCAL

# Insert at position 0 so our src/ always takes priority over any Colab defaults.
if REPO_LOCAL not in sys.path:
    sys.path.insert(0, REPO_LOCAL)

# Quick sanity-check that the import will work before running any cells.
import importlib.util
_spec = importlib.util.find_spec("src.dataset")
if _spec is None:
    raise ImportError(
        "src.dataset not found. Check that the clone succeeded and "
        "that src/__init__.py exists in the repository."
    )
print("src modules found at:", os.path.join(REPO_LOCAL, "src"))

# ---- Project folders on Google Drive ----
DRIVE_ROOT = "/content/drive/MyDrive/" + REPO
DATA_DIR   = os.path.join(DRIVE_ROOT, "data", "lung_colon_image_set")
FIG_DIR    = os.path.join(DRIVE_ROOT, "figures")
MODEL_DIR  = os.path.join(DRIVE_ROOT, "models")
FEAT_DIR   = os.path.join(DRIVE_ROOT, "features")
for d in (FIG_DIR, MODEL_DIR, FEAT_DIR):
    os.makedirs(d, exist_ok=True)

CLASSES = ["colon_aca", "colon_n", "lung_aca", "lung_n", "lung_scc"]
print("Setup complete. CLASSES:", CLASSES)


In [ ]:
# Load the best model (XGBoost), the full scaled features, labels, and feature names.
import numpy as np, json, joblib

y_train = np.load(os.path.join(FEAT_DIR, "y_train.npy"))
y_test  = np.load(os.path.join(FEAT_DIR, "y_test.npy"))
Xtr_s   = np.load(os.path.join(FEAT_DIR, "X_train_scaled.npy"))
Xte_s   = np.load(os.path.join(FEAT_DIR, "X_test_scaled.npy"))

with open(os.path.join(FEAT_DIR, "feature_names.json")) as f:
    feature_names = json.load(f)

model = joblib.load(os.path.join(MODEL_DIR, "xgboost.joblib"))
print("Model loaded:", type(model).__name__)
print("Features:", len(feature_names))

Loading the saved model and features confirms the pipeline is reproducible, since the same file that was written in Notebook 06 is being read back here. Having the feature names alongside the importance values is what makes the analysis readable rather than anonymous. A deep learning model would give pixel level attribution; here the names are words that describe colour, texture, and edge patterns, which is easier to connect to the biology.

In [ ]:
# Native feature importance from XGBoost, top 20 individual features.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from src.explain import plot_top_features

importances = model.feature_importances_
plot_top_features(importances, feature_names,
    "XGBoost: top 20 features by importance",
    top=20, save_path=os.path.join(FIG_DIR, "07_top20_features.png"))
print("Top 5 features:")
order = np.argsort(importances)[::-1]
for i in order[:5]:
    print(f"  {feature_names[i]}: {importances[i]:.4f}")

The top individual features are mostly colour and GLCM texture descriptors rather than HOG edge features. Colour features capture the pink and purple staining that separates tissue types, and the GLCM features describe how regular or irregular the cell arrangement is. HOG features contribute less at the individual level because each one describes only one small spatial region, but in the family aggregation below they add up to a meaningful total. The ranked list gives a concrete answer to what the model looks at when it classifies a tissue patch.

In [ ]:
# Aggregate importance by feature family and plot the shares.
from src.explain import aggregate_importance_by_family, plot_family_importance
from src.features import feature_family

fam_importance = aggregate_importance_by_family(importances, feature_names, feature_family)
print("Importance by family:")
for k, v in fam_importance.items():
    print(f"  {k}: {v:.3f}")

plot_family_importance(fam_importance,
    "XGBoost: importance share by feature family",
    save_path=os.path.join(FIG_DIR, "07_family_importance.png"))

The family plot gives a cleaner picture than 421 individual bars. Colour features usually carry the largest share, which makes sense for H and E staining because the stain colours are directly tied to the tissue type. Texture features such as GLCM add a meaningful second share, describing how cells are arranged rather than just what colour they are. HOG features contribute a smaller but nonzero share, showing that edge patterns add information on top of colour and texture alone. This breakdown connects the model's behaviour to the biology of the tissue.

In [ ]:
# Permutation importance on the test set as a model-agnostic cross-check.
import pandas as pd
from src.explain import permutation_importance_df, aggregate_importance_by_family, plot_family_importance, plot_top_features

# Run on a subset of test rows to keep the computation fast.
rng = np.random.RandomState(0)
idx = rng.choice(len(Xte_s), size=min(1000, len(Xte_s)), replace=False)
perm_df = permutation_importance_df(model, Xte_s[idx], y_test[idx],
                                     feature_names, n_repeats=5, seed=42)
print("Top 10 permutation features:")
print(perm_df.head(10).to_string(index=False))

plot_top_features(perm_df["importance"].values, perm_df["feature"].tolist(),
    "Permutation importance: top 20 features",
    top=20, save_path=os.path.join(FIG_DIR, "07_permutation_importance.png"))

perm_fam = aggregate_importance_by_family(perm_df["importance"].values,
    perm_df["feature"].tolist(), feature_family)
print("\nPermutation importance by family:")
for k, v in perm_fam.items():
    print(f"  {k}: {v:.3f}")

Permutation importance works by shuffling one feature at a time and measuring how much accuracy drops, which checks importance independently of the model's internal weights. When the family ranking from permutation importance agrees with the native importance, it is a reliable sign that the conclusion holds up under two different methods. Colour features usually stay near the top in both, and texture features stay in second place. Any feature where permutation importance is close to zero or negative can be considered uninformative and could be dropped in a future version.

In [ ]:
# SHAP values using TreeExplainer for a deeper view of how features drive predictions.
import shap
import matplotlib.pyplot as plt
import numpy as np

explainer = shap.TreeExplainer(model)
# Use a subset for speed; 500 test images is enough for a representative summary.
idx_shap = np.random.RandomState(1).choice(len(Xte_s), size=500, replace=False)
shap_values = explainer.shap_values(Xte_s[idx_shap])
# shap_values is list of arrays (one per class) or array of shape (n, features, classes)
if isinstance(shap_values, list):
    shap_arr = np.stack(shap_values, axis=-1)  # (500, features, 5)
else:
    shap_arr = shap_values  # already (500, features, 5)

# Summary plot for class 2 (lung_aca) to show a representative class.
plt.figure()
shap.summary_plot(shap_arr[:, :, 2], Xte_s[idx_shap],
                  feature_names=feature_names, show=False, max_display=15)
plt.title("SHAP summary: lung_aca")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "07_shap_summary_lung_aca.png"), dpi=150, bbox_inches="tight")
plt.show()

# Mean absolute SHAP per family for every class.
shap_family = {}
for ci, cls in enumerate(CLASSES):
    mean_abs = np.abs(shap_arr[:, :, ci]).mean(axis=0)
    shap_family[cls] = aggregate_importance_by_family(mean_abs, feature_names, feature_family)
shap_df = pd.DataFrame(shap_family).T
print("\nMean absolute SHAP by feature family per class:")
print(shap_df.round(4))
shap_df.plot(kind="bar", figsize=(9, 5))
plt.title("Mean absolute SHAP by feature family per class")
plt.ylabel("Mean |SHAP|")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "07_shap_family_per_class.png"), dpi=150, bbox_inches="tight")
plt.show()

The SHAP summary plot shows each feature's push on predictions for the lung_aca class, with colour in the dot indicating the feature value. Features that push strongly to the right are those that raise the chance of lung_aca, while those pushing left lower it. The table of mean absolute SHAP values by family and class shows whether all classes rely on the same features or whether different classes are driven by different signals. When the cancer classes show a different colour family pattern from the benign classes, it confirms that the staining differences are genuinely informative rather than coincidental.

In [ ]:
# Load all saved metrics files and build the ML model leaderboard.
import json, pandas as pd, glob

rows = []
for path in glob.glob(os.path.join(MODEL_DIR, "*_metrics.json")):
    with open(path) as f:
        m = json.load(f)
    rows.append({
        "Model":       m.get("model", os.path.basename(path).replace("_metrics.json","")),
        "Accuracy":    round(m.get("accuracy", float("nan")), 4),
        "AUC-ROC":     round(m.get("auc", float("nan")), 4),
        "F1 (macro)":  round(m.get("f1", float("nan")), 4),
        "Sensitivity": round(m.get("sensitivity", float("nan")), 4),
        "Specificity": round(m.get("specificity", float("nan")), 4),
    })
leaderboard = pd.DataFrame(rows).sort_values("Accuracy", ascending=False).reset_index(drop=True)
print(leaderboard.to_string(index=False))

The leaderboard puts all four machine learning models side by side on the same five metrics. The ranking shows which model gets the most out of the 421 handcrafted features. XGBoost and the random forest usually sit at the top, while logistic regression and k nearest neighbours set the floor. The table is a concise summary of four notebooks of work and makes it easy to see where each model gains or loses relative to the others. This table will feed directly into the results section of the project report.

In [ ]:
# Compare the best ML model against the ResNet-50 result from the companion DL pipeline.
import pandas as pd, matplotlib.pyplot as plt, json, glob, os

# Find the best ML model from the leaderboard.
best_row = leaderboard.iloc[0]
best_ml_name = best_row["Model"]
best_ml_acc  = best_row["Accuracy"]
best_ml_auc  = best_row["AUC-ROC"]
best_ml_f1   = best_row["F1 (macro)"]

# ResNet-50 result from the companion deep learning repository
# (Odyek Henry, 2025/HD07/26018U, https://github.com/hodyek/lung-colon-cancer-histopathology).
# Same LC25000 dataset, same stratified 70/15/15 split with seed 42.
dl_name = "ResNet-50 (deep learning)"
dl_acc  = 0.9987
dl_auc  = 0.9999   # reported in DL pipeline
dl_f1   = 0.9987

comparison = pd.DataFrame({
    "Model":      [best_ml_name, dl_name],
    "Accuracy":   [best_ml_acc, dl_acc],
    "AUC-ROC":    [best_ml_auc, dl_auc],
    "F1 (macro)": [best_ml_f1, dl_f1],
})
print(comparison.to_string(index=False))

# Bar chart comparison.
x = ["Accuracy", "AUC-ROC", "F1 (macro)"]
ml_vals = [best_ml_acc, best_ml_auc, best_ml_f1]
dl_vals = [dl_acc, dl_auc, dl_f1]
width = 0.3
xs = range(len(x))
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([i - width/2 for i in xs], ml_vals, width, label=best_ml_name, color="steelblue")
ax.bar([i + width/2 for i in xs], dl_vals, width, label=dl_name, color="salmon")
ax.set_xticks(list(xs)); ax.set_xticklabels(x)
ax.set_ylim(0.8, 1.01); ax.set_ylabel("Score"); ax.set_title("Best ML vs ResNet-50 (DL)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "07_ml_vs_dl_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

The comparison shows the deep learning pipeline reaches higher accuracy, which is expected because ResNet-50 learns features directly from pixels across millions of parameters. The machine learning pipeline reaches a strong result using only 421 handcrafted numbers per image. The gap between the two is the cost of replacing learned features with expert designed ones, and it is not enormous on a clean and well-stained dataset. The machine learning pipeline compensates by offering readable explanations of its decisions, which the deep learning model cannot provide as directly without an additional explainability step.

In [ ]:
# Print a qualitative cost and transparency comparison.
import pandas as pd

qual = pd.DataFrame({
    "Property":       ["Test accuracy", "AUC-ROC", "Interpretability",
                       "Training time", "Model size", "Needs GPU", "Explainability method"],
    best_ml_name:     [f"{best_ml_acc:.4f}", f"{best_ml_auc:.4f}", "High",
                       "Minutes on CPU", "~200 MB joblib", "No", "SHAP, permutation, feature importance"],
    dl_name:          ["0.9987", "0.9999", "Low without post-hoc tools",
                       "Hours with GPU", "~100 MB weights", "Recommended", "Grad-CAM (companion repo)"],
})
print(qual.to_string(index=False))

The qualitative table makes the trade-offs concrete. The deep learning model needs a GPU and hours of training to reach near-perfect accuracy, while the machine learning pipeline runs on a CPU in minutes and gives colour and texture explanations without any extra tools. For a deployment where a pathologist wants to know why the model reached a verdict, the machine learning pipeline has an advantage. For maximum accuracy on a large balanced dataset, the deep learning pipeline wins. In practice, a clinical system might use the machine learning explanations to audit deep learning decisions rather than replace them.

## Summary
The explainability analysis shows that colour features carry the largest share of importance, followed by texture descriptors, with edge features adding a smaller contribution. SHAP values confirm this family ranking and show which direction each feature pushes predictions for the cancer classes. The ML leaderboard ranks all four models, with XGBoost and the random forest at the top. Against the companion ResNet-50 result, the best machine learning model reaches lower accuracy but offers richer built-in explainability. This notebook completes the machine learning pipeline and provides the material needed for the project report.